# 04e — EXP_05: RAGAS Evaluation (Claude Sonnet 4.6 judge — all 5 metrics)

Scores `results/exp_05_multi_hop_rag__golden_234/predictions.jsonl` against the 234-row golden subset.

**Headline comparison numbers** (Phase 4 final cross-architecture):

| Architecture | Acuuracy (test_1273) | Faithfulness | Hallucination Rate | Context Precision | Context Recall |
|---|---:|---:|---:|---:|---:|
| EXP_01 No-RAG | 0.7738 | n/a | n/a | n/a | n/a |
| EXP_02 Naive Dense | 0.7573 | 0.1308 | 0.8957 | 0.3285 | 0.4124 |
| EXP_03 Sparse BM25 | 0.7581 | (pending) | (pending) | (pending) | (pending) |
| EXP_04 Hybrid | (pending) | (pending) | (pending) | (pending) | (pending) |
| **EXP_05 Multi-Hop** | **??** | **??** | **??** | **??** | **??** |

**Falsifiable hypothesis** (anchored in [`docs/output_notes/04b_exp02_output.md` §3.5](../docs/output_notes/04b_exp02_output.md)): *Multi-Hop achieves Faithfulness > 0 on the 13 multi-hop golden rows where Naive scored 0.000. Falsified if Multi-Hop Faithfulness on `requires_multihop=yes` rows ≤ 0.05.*

Stages: smoke 10 (~$0.50, ~1 min) → full 234 (~$11–12, ~10–20 min) → NaN rescore if needed.

Note: EXP_05 retrieval returns up to 15 chunks per question (vs 5 for EXP_02–04), so RAGAS prompts are longer. Cost may be slightly higher than EXP_02–04 (~$13–15) due to the bigger context.

## 1. Setup

In [1]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.eval.ragas_eval import score_predictions

for k in ["ANTHROPIC_API_KEY"]:
    print(f"{k}: {'\u2713 present' if os.environ.get(k) else '\u2717 MISSING'}")
print("Repo root:", REPO_ROOT)

ANTHROPIC_API_KEY: ✓ present
Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project


## 2. Configuration

In [5]:
PRED_DIR = REPO_ROOT / "results" / "exp_05_multi_hop_rag__golden_234"
GOLDEN_PATH = REPO_ROOT / "data" / "processed" / "golden_ragas_300.jsonl"
CHUNKS_PATH = REPO_ROOT / "data" / "processed" / "chunks.parquet"

RUN_SMOKE = True
SMOKE_N = 10
RUN_FULL = True
RUN_RESCORE_NANS = False
OVERWRITE_SCORES = False

print("Predictions:", PRED_DIR)
print("Golden     :", GOLDEN_PATH)
print("Chunks     :", CHUNKS_PATH)

assert PRED_DIR.exists(), (
    f"{PRED_DIR} doesn't exist — run 04e_exp05_multi_hop_rag.ipynb Stage B first."
)

Predictions: /Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_05_multi_hop_rag__golden_234
Golden     : /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/golden_ragas_300.jsonl
Chunks     : /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/chunks.parquet


## 3. Stage A — Smoke (10 rows × 5 metrics · ~1 min · ~$0.50–0.80)

In [3]:
import pandas as pd
import shutil

if RUN_SMOKE:
    smoke_dir = PRED_DIR.parent / "exp_05_multi_hop_rag__golden_234_ragas_smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    for fname in ("predictions.jsonl", "retrieval.jsonl", "summary.json"):
        shutil.copy2(PRED_DIR / fname, smoke_dir / fname)

    smoke_summary = score_predictions(
        predictions_dir=smoke_dir,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=SMOKE_N,
        overwrite=OVERWRITE_SCORES,
    )
    print('\n=== Stage A smoke summary (10 rows × 5 metrics) ===')
    for k, v in smoke_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")):
            print(f"  {k:30s} : {v}")
    smoke_scores = pd.read_csv(smoke_dir / "ragas_scores.csv")
    cols = [c for c in smoke_scores.columns if c in ("question_id", "_is_correct", "faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness")]
    print("\n=== Per-row scores (smoke 10) ===")
    print(smoke_scores[cols].to_string(index=False))
else:
    print("RUN_SMOKE = False — skipped.")

[ragas] joined 10 rows | metrics = ['context_recall', 'context_precision', 'faithfulness', 'answer_correctness', 'answer_relevancy']


/Users/rajak/Workstation/Projects/myGitHub/thesis-project/src/eval/ragas_eval.py:240: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=embed_model))


[ragas] RunConfig: max_workers=4, max_retries=10, max_wait=120s, timeout=180s


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

[ragas] judge wall-time 269.6 s for 10 rows × 5 metrics
[ragas] wrote /Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_05_multi_hop_rag__golden_234_ragas_smoke/ragas_scores.csv

=== Stage A smoke summary (10 rows × 5 metrics) ===
  RAGAS_Faithfulness             : 0.19595238095238096
  RAGAS_Hallucination_Rate       : 0.8
  RAGAS_Answer_Relevance         : 0.6064238492290107
  RAGAS_Context_Precision        : 0.24147007620952357
  RAGAS_Context_Recall           : 0.7
  Answer_Correctness             : 0.8207337041870361
  ragas_metrics_run              : ['context_recall', 'context_precision', 'faithfulness', 'answer_correctness', 'answer_relevancy']
  ragas_n_scored                 : 10
  ragas_judge                    : claude-sonnet-4-6
  ragas_timestamp_utc            : 2026-05-10T06:50:03Z

=== Per-row scores (smoke 10) ===
 context_recall  context_precision  faithfulness  answer_correctness  answer_relevancy question_id  _is_correct
            1.0           

**Acceptance gates (Stage A)** — same as EXP_04: no errors, all 5 metric cols present, NaN < 30 %, Faithfulness ≥ 0 on at least some rows, AC correlates with `_is_correct`.

## 4. Stage B — Full 234 (~10–20 min · ~$13–15)

Multi-Hop's larger context (up to 15 chunks vs 5) makes RAGAS prompts longer than EXP_02–04. Cost projection is slightly higher; user can also stop after Stage A if budget tight.

In [6]:
if RUN_FULL:
    full_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=None,
        overwrite=OVERWRITE_SCORES,
    )
    print("\n=== Stage B full summary (234 rows × 5 metrics) ===")
    for k, v in full_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")) or k == "Acuuracy":
            print(f"  {k:30s} : {v}")
else:
    print("RUN_FULL = False")

[ragas] joined 234 rows | metrics = ['context_recall', 'context_precision', 'faithfulness', 'answer_correctness', 'answer_relevancy']
[ragas] RunConfig: max_workers=4, max_retries=10, max_wait=120s, timeout=180s


Evaluating:   0%|          | 0/1170 [00:00<?, ?it/s]

Exception raised in Job[171]: TimeoutError()
Exception raised in Job[374]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[659]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[906]: TimeoutError()
Exception raised in Job[911]: TimeoutError()
Exception raised in Job[916]: TimeoutError()
Exception raised in Job[918]: TimeoutError()
Exception raised in Job[921]: TimeoutError()
Exception raised in Job[926]: TimeoutError()
Exception raised in Job[931]: TimeoutError()
Exception raised in Job[932]: TimeoutError()
Exception raised in Job[936]: TimeoutError()
Exception raised in Job[937]: TimeoutError()
Exception raised in Job[951]: TimeoutError()


[ragas] judge wall-time 12157.0 s for 234 rows × 5 metrics
[ragas] wrote /Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_05_multi_hop_rag__golden_234/ragas_scores.csv

=== Stage B full summary (234 rows × 5 metrics) ===
  Acuuracy                       : 0.9017094017094017
  RAGAS_Faithfulness             : 0.28331669527790215
  RAGAS_Hallucination_Rate       : 0.7370689655172413
  RAGAS_Answer_Relevance         : 0.595453566104136
  RAGAS_Context_Precision        : 0.3737426338573044
  RAGAS_Context_Recall           : 0.7115384615384616
  Answer_Correctness             : 0.8684697826560158
  ragas_metrics_run              : ['context_recall', 'context_precision', 'faithfulness', 'answer_correctness', 'answer_relevancy']
  ragas_n_scored                 : 234
  ragas_judge                    : claude-sonnet-4-6
  ragas_timestamp_utc            : 2026-05-10T10:13:10Z


## 5. Stage C — NaN rescore

In [ ]:
if RUN_RESCORE_NANS:
    rescored_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        rescore_nans=True,
    )
    df = pd.read_csv(PRED_DIR / "ragas_scores.csv")
    for col in ("faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness"):
        if col in df.columns:
            n_nan = pd.to_numeric(df[col], errors="coerce").isna().sum()
            print(f"  {col:25s}: {n_nan} NaN remaining of {len(df)}")
else:
    print("RUN_RESCORE_NANS = False")

## 6. Inspect — does Multi-Hop achieve Faithfulness > 0 on multi-hop rows?

In [7]:
scores_path = PRED_DIR / "ragas_scores.csv"
if scores_path.exists():
    df = pd.read_csv(scores_path)
    print(f"\nrows: {len(df)}")

    # Cross-architecture overall comparison (golden 234)
    print('\n--- EXP_05 vs other architectures (golden 234) ---')
    e2 = json.loads((REPO_ROOT / 'results' / 'exp_02_naive_rag__golden_234' / 'summary.json').read_text())
    for col, key in [('faithfulness', 'RAGAS_Faithfulness'), ('context_precision', 'RAGAS_Context_Precision'), ('context_recall', 'RAGAS_Context_Recall'), ('answer_relevancy', 'RAGAS_Answer_Relevance'), ('answer_correctness', 'Answer_Correctness')]:
        if col not in df.columns: continue
        s = pd.to_numeric(df[col], errors='coerce')
        e5_mean = s.mean()
        e2_mean = e2.get(key)
        delta_s = f'{(e5_mean - e2_mean)*100:+.2f} pp' if isinstance(e2_mean, (int, float)) else 'n/a'
        print(f'  {col:25s}: EXP_05={e5_mean:.4f}  EXP_02={e2_mean if e2_mean else "--":>8}  Δ={delta_s}  NaN={s.isna().sum()}')

    # Multi-hop-specific subset — where the falsifiable hypothesis is tested
    golden_rows = [json.loads(l) for l in open(REPO_ROOT / 'data' / 'processed' / 'golden_ragas_300.jsonl')]
    mh_qids = {f"golden_{r['question_id']:03d}" for r in golden_rows if r.get('requires_multihop') == 'yes'}
    mh_df = df[df['question_id'].isin(mh_qids)]
    print(f'\n  Multi-hop subset (requires_multihop=yes): n={len(mh_df)}')
    for col in ('faithfulness', 'context_precision', 'context_recall', 'answer_correctness'):
        if col in mh_df.columns:
            s = pd.to_numeric(mh_df[col], errors='coerce')
            print(f'    {col:25s}: mean={s.mean():.4f}  (EXP_02 had F=0.000 on these rows)')
    if 'faithfulness' in mh_df.columns:
        f_mh = pd.to_numeric(mh_df['faithfulness'], errors='coerce').mean()
        verdict = '\u2713 SUPPORTED' if f_mh > 0.05 else '\u2717 FALSIFIED'
        print(f'\n  Hypothesis: Multi-Hop Faithfulness > 0.05 on multi-hop rows? → {verdict}  (got {f_mh:.4f})')
else:
    print("No ragas_scores.csv yet — run Stage B first.")


rows: 234

--- EXP_05 vs other architectures (golden 234) ---
  faithfulness             : EXP_05=0.2833  EXP_02=0.1308175399479747  Δ=+15.25 pp  NaN=2
  context_precision        : EXP_05=0.3737  EXP_02=0.328532608674367  Δ=+4.52 pp  NaN=9
  context_recall           : EXP_05=0.7115  EXP_02=0.41239316239316237  Δ=+29.91 pp  NaN=0
  answer_relevancy         : EXP_05=0.5955  EXP_02=0.5960520825068125  Δ=-0.06 pp  NaN=2
  answer_correctness       : EXP_05=0.8685  EXP_02=0.8376479070859398  Δ=+3.08 pp  NaN=1

  Multi-hop subset (requires_multihop=yes): n=13
    faithfulness             : mean=0.2291  (EXP_02 had F=0.000 on these rows)
    context_precision        : mean=0.3573  (EXP_02 had F=0.000 on these rows)
    context_recall           : mean=0.6154  (EXP_02 had F=0.000 on these rows)
    answer_correctness       : mean=0.7688  (EXP_02 had F=0.000 on these rows)

  Hypothesis: Multi-Hop Faithfulness > 0.05 on multi-hop rows? → ✓ SUPPORTED  (got 0.2291)


---

**Next.** With all 5 baseline + 5 RAGAS evaluations done, write the **cross-architecture analysis** for Phase 4 — Table 1 of the thesis. The 4-RAG matrix (Naive · Sparse · Hybrid · Multi-Hop) is now fully populated, ready for the discussion-chapter narrative.